# BenchmarkData2 Analysis - Enhanced Dataset with Fuel Gas Molecular Weight

This notebook analyzes the enhanced dataset (BenchmarkData2.xlsx) which includes the molecular weight of fuel gas. This additional parameter will enable us to calculate critical combustion properties and create physics-informed features for improved furnace control modeling.

## Key Enhancements in This Dataset:
1. **Fuel Gas Molecular Weight** - Enables density calculations
2. **Physics-informed features** - Stoichiometric calculations, Wobbe index
3. **Enhanced combustion modeling** - Real heat input calculations
4. **Time indexing** - Sequential minute-based indexing for better temporal analysis

## Objectives:
1. Load and preprocess the enhanced dataset
2. Calculate fuel gas density and combustion properties
3. Create physics-informed features
4. Prepare data for improved LNN model training
5. Compare with previous dataset performance

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for publication quality
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 12

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)

print("🚀 BENCHMARKDATA2 ANALYSIS SETUP")
print("=" * 50)
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Analysis ready for enhanced dataset with molecular weight!")
print("=" * 50)

🚀 BENCHMARKDATA2 ANALYSIS SETUP
NumPy version: 1.26.4
Pandas version: 2.2.2
Analysis ready for enhanced dataset with molecular weight!


## 1. Data Loading and Initial Exploration

Load the enhanced dataset from BenchmarkData2.xlsx with molecular weight information and examine its structure.

In [8]:
# Load BenchmarkData2.xlsx with molecular weight information
print("📊 LOADING ENHANCED DATASET")
print("=" * 50)

# Load the data from the specified sheet
data_file = '/Users/abuhuzaifahbidin/Documents/GitHub/furnace-commander/backend/data/BenchmarkData2.xlsx'

try:
    # Load data from 'data2' sheet
    df_raw = pd.read_excel(data_file, sheet_name='Data2')
    print(f"✅ Data loaded successfully from BenchmarkData2.xlsx")
    print(f"Original shape: {df_raw.shape}")
    
except FileNotFoundError:
    print(f"❌ File not found: {data_file}")
    print("Please ensure BenchmarkData2.xlsx is in the data folder")
except Exception as e:
    print(f"❌ Error loading data: {e}")

# Display basic information about the dataset
print(f"\n📋 DATASET OVERVIEW")
print("=" * 50)
print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")

print(f"\n📝 COLUMN NAMES")
print("=" * 50)
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i:2d}. {col}")

# Display first few rows
print(f"\n👁️  FIRST 5 ROWS")
print("=" * 50)
print(df_raw.head())

# Check for the molecular weight column
if 'MW' in df_raw.columns or 'MolecularWeight' in df_raw.columns or 'Molecular Weight' in df_raw.columns:
    print(f"\n✅ Molecular weight column found!")
else:
    print(f"\n⚠️  Searching for molecular weight column...")
    mw_candidates = [col for col in df_raw.columns if 'mw' in col.lower() or 'molecular' in col.lower() or 'weight' in col.lower()]
    if mw_candidates:
        print(f"Potential MW columns: {mw_candidates}")
    else:
        print("No obvious molecular weight column found. Will examine data further.")

print(f"\nReady for data transformation! 🚀")

📊 LOADING ENHANCED DATASET
✅ Data loaded successfully from BenchmarkData2.xlsx
Original shape: (171064, 8)

📋 DATASET OVERVIEW
Rows: 171064
Columns: 8

📝 COLUMN NAMES
 1. Date
 2. InletTemp
 3. InletFlow
 4. OutletTemp
 5. AirFuelRatio
 6. FuelGasFlow
 7. ExcessO2
 8. FuelGasMW

👁️  FIRST 5 ROWS
          Date  InletTemp  InletFlow OutletTemp AirFuelRatio FuelGasFlow  \
0        45717 261.921478 165.248749 289.977783    11.498828  588.381287   
1 45717.000694 261.911652 164.872009 290.056610    11.517564  586.792297   
2 45717.001389 261.878510 164.925217 290.141907    11.536301  584.424438   
3 45717.002083 261.934540 164.916031 290.285034    11.555037  582.956848   
4 45717.002778 261.991547 165.054184 290.246033    11.576279  580.167053   

  ExcessO2 FuelGasMW  
0 2.299289 12.686572  
1 2.292834 12.686572  
2 2.336514 12.686572  
3 2.293091 12.870671  
4 2.144124 12.739172  

⚠️  Searching for molecular weight column...
Potential MW columns: ['FuelGasMW']

Ready for data transforma

In [9]:
# Data Cleaning and Missing Value Analysis
print("🧹 DATA CLEANING AND VALIDATION")
print("=" * 60)

# Check for missing values
print(f"📊 MISSING VALUES ANALYSIS")
print("=" * 50)
missing_summary = df_raw.isnull().sum()
missing_percent = (missing_summary / len(df_raw)) * 100

for col in df_raw.columns:
    missing_count = missing_summary[col]
    missing_pct = missing_percent[col]
    if missing_count > 0:
        print(f"❌ {col}: {missing_count:,} missing ({missing_pct:.2f}%)")
    else:
        print(f"✅ {col}: No missing values")

# Check for empty string values and other issues
print(f"\n🔍 DATA TYPE AND CONTENT ANALYSIS")
print("=" * 50)

for col in df_raw.columns:
    if df_raw[col].dtype == 'object':
        print(f"⚠️  {col}: object type - checking content...")
        
        # Check for empty strings
        empty_strings = (df_raw[col] == '').sum()
        if empty_strings > 0:
            print(f"   - Empty strings: {empty_strings}")
        
        # Check for whitespace-only strings
        if df_raw[col].dtype == 'object':
            whitespace_only = df_raw[col].astype(str).str.strip().eq('').sum()
            if whitespace_only > 0:
                print(f"   - Whitespace-only: {whitespace_only}")
        
        # Show unique non-numeric values (first few)
        try:
            non_numeric = df_raw[col][pd.to_numeric(df_raw[col], errors='coerce').isna()]
            if len(non_numeric) > 0:
                unique_non_numeric = non_numeric.unique()[:5]  # First 5 unique non-numeric values
                print(f"   - Non-numeric values (sample): {unique_non_numeric}")
        except:
            pass
    else:
        print(f"✅ {col}: {df_raw[col].dtype}")

# Check for completely empty rows
print(f"\n🔍 EMPTY ROWS ANALYSIS")
print("=" * 50)
empty_rows = df_raw.isnull().all(axis=1).sum()
print(f"Completely empty rows: {empty_rows}")

if empty_rows > 0:
    # Find where empty rows start
    empty_row_indices = df_raw[df_raw.isnull().all(axis=1)].index
    if len(empty_row_indices) > 0:
        first_empty = empty_row_indices[0]
        print(f"First empty row at index: {first_empty}")
        print(f"Rows after first empty: {len(df_raw) - first_empty}")

# Check for rows where all numeric columns are NaN or empty
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    numeric_empty_rows = df_raw[numeric_cols].isnull().all(axis=1).sum()
    print(f"Rows with all numeric columns empty: {numeric_empty_rows}")

# Show data range for analysis
print(f"\n📈 DATA RANGE ANALYSIS")
print("=" * 50)
print(f"Total rows: {len(df_raw):,}")
print(f"Data at row 170,000: ")
try:
    print(df_raw.iloc[170000:170005])
except:
    print("Row 170,000 not available")

print(f"\nLast 5 rows:")
print(df_raw.tail())

print(f"\n✅ Data cleaning analysis completed!")

🧹 DATA CLEANING AND VALIDATION
📊 MISSING VALUES ANALYSIS
❌ Date: 4,776 missing (2.79%)
❌ InletTemp: 4,776 missing (2.79%)
❌ InletFlow: 4,775 missing (2.79%)
❌ OutletTemp: 4,774 missing (2.79%)
❌ AirFuelRatio: 4,772 missing (2.79%)
❌ FuelGasFlow: 4,771 missing (2.79%)
❌ ExcessO2: 4,770 missing (2.79%)
✅ FuelGasMW: No missing values

🔍 DATA TYPE AND CONTENT ANALYSIS
⚠️  Date: object type - checking content...
   - Non-numeric values (sample): ['Resize to show all values' nan]
⚠️  InletTemp: object type - checking content...
   - Non-numeric values (sample): ['I/O Timeout' 'Bad' 'Resize to show all values' nan]
⚠️  InletFlow: object type - checking content...
   - Non-numeric values (sample): ['I/O Timeout' 'Bad' 'Resize to show all values' nan]
⚠️  OutletTemp: object type - checking content...
   - Non-numeric values (sample): ['I/O Timeout' 'Bad' nan]
⚠️  AirFuelRatio: object type - checking content...
   - Non-numeric values (sample): ['I/O Timeout' 'Bad' nan]
⚠️  FuelGasFlow: object t

In [10]:
# Clean the Data - Remove Empty Rows and Convert Data Types
print("🔧 DATA CLEANING IMPLEMENTATION")
print("=" * 60)

# Start with a copy of the raw data
df_cleaned = df_raw.copy()

print(f"Original shape: {df_cleaned.shape}")

# Step 1: Remove completely empty rows
empty_rows_mask = df_cleaned.isnull().all(axis=1)
df_cleaned = df_cleaned[~empty_rows_mask]
print(f"After removing completely empty rows: {df_cleaned.shape}")

# Step 2: Remove rows where critical columns are missing or empty
# Identify critical columns (all except Date)
critical_columns = [col for col in df_cleaned.columns if col not in ['Date']]

# Convert object columns to numeric, coercing errors to NaN
for col in critical_columns:
    if df_cleaned[col].dtype == 'object':
        print(f"Converting {col} from object to numeric...")
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Remove rows where any critical column is NaN
rows_before = len(df_cleaned)
df_cleaned = df_cleaned.dropna(subset=critical_columns)
rows_removed = rows_before - len(df_cleaned)
print(f"Removed {rows_removed:,} rows with missing critical data")
print(f"After cleaning: {df_cleaned.shape}")

# Step 3: Reset index
df_cleaned = df_cleaned.reset_index(drop=True)

# Step 4: Validate data ranges (remove obvious outliers)
print(f"\n📊 DATA VALIDATION")
print("=" * 50)

# Check for reasonable ranges
validation_ranges = {
    'InletTemp': (200, 400),      # Reasonable inlet temperature range (°C)
    'InletFlow': (0, 1000),       # Reasonable flow range
    'OutletTemp': (250, 500),     # Reasonable outlet temperature range (°C)
    'AirFuelRatio': (5, 25),      # Reasonable AFR range
    'FuelGasFlow': (0, 2000),     # Reasonable fuel flow range
    'ExcessO2': (0, 10),          # Reasonable excess O2 range (%)
    'FuelGasMW': (10, 30),        # Reasonable molecular weight range for fuel gas
}

outliers_removed = 0
for col, (min_val, max_val) in validation_ranges.items():
    if col in df_cleaned.columns:
        before_count = len(df_cleaned)
        df_cleaned = df_cleaned[(df_cleaned[col] >= min_val) & (df_cleaned[col] <= max_val)]
        removed = before_count - len(df_cleaned)
        outliers_removed += removed
        if removed > 0:
            print(f"⚠️  {col}: Removed {removed:,} outliers (outside {min_val}-{max_val})")
        else:
            print(f"✅ {col}: No outliers found")

print(f"\nTotal outliers removed: {outliers_removed:,}")
print(f"Final cleaned shape: {df_cleaned.shape}")

# Step 5: Final data summary
print(f"\n📈 CLEANED DATA SUMMARY")
print("=" * 60)
print(df_cleaned.describe())

print(f"\n🔍 DATA TYPES AFTER CLEANING")
print("=" * 50)
print(df_cleaned.dtypes)

print(f"\n👁️  CLEANED DATA SAMPLE")
print("=" * 50)
print(df_cleaned.head())

# Check for any remaining issues
remaining_nulls = df_cleaned.isnull().sum().sum()
if remaining_nulls == 0:
    print(f"\n✅ Data cleaning completed successfully!")
    print(f"   - Original rows: {len(df_raw):,}")
    print(f"   - Cleaned rows: {len(df_cleaned):,}")
    print(f"   - Rows removed: {len(df_raw) - len(df_cleaned):,}")
    print(f"   - Data quality: Ready for analysis!")
else:
    print(f"\n⚠️  Warning: {remaining_nulls} null values still remain")

# Update df_raw for next steps
df_raw = df_cleaned.copy()
print(f"\n🚀 Ready for transformation with clean data!")

🔧 DATA CLEANING IMPLEMENTATION
Original shape: (171064, 8)
After removing completely empty rows: (171064, 8)
Converting InletTemp from object to numeric...
Converting InletFlow from object to numeric...
Converting OutletTemp from object to numeric...
Converting AirFuelRatio from object to numeric...
Converting FuelGasFlow from object to numeric...
Converting ExcessO2 from object to numeric...
Converting FuelGasMW from object to numeric...
Removed 4,793 rows with missing critical data
After cleaning: (166271, 8)

📊 DATA VALIDATION
✅ InletTemp: No outliers found
✅ InletFlow: No outliers found
✅ OutletTemp: No outliers found
✅ AirFuelRatio: No outliers found
✅ FuelGasFlow: No outliers found
✅ ExcessO2: No outliers found
⚠️  FuelGasMW: Removed 599 outliers (outside 10-30)

Total outliers removed: 599
Final cleaned shape: (165672, 8)

📈 CLEANED DATA SUMMARY
          InletTemp     InletFlow    OutletTemp  AirFuelRatio   FuelGasFlow  \
count 165672.000000 165672.000000 165672.000000 165672.0

In [14]:
# Data Transformation: Time Indexing and Column Standardization
print("🔄 DATA TRANSFORMATION")
print("=" * 50)

# Create a copy for transformation
df = df_raw.copy()

# Replace date/time column with sequential minute indexing
print(f"Original columns with potential time data:")
time_candidates = [col for col in df.columns if any(keyword in col.lower() 
                  for keyword in ['date', 'time', 'timestamp', 'datetime'])]
print(f"Time candidates: {time_candidates}")

# Create sequential minute index starting from 1
df['TimeIndex'] = range(1, len(df) + 1)
print(f"✅ Created TimeIndex column: 1 to {len(df)} minutes")

# Drop original date/time columns if they exist
if time_candidates:
    df = df.drop(columns=time_candidates)
    print(f"✅ Dropped original time columns: {time_candidates}")

# Standardize column names (remove spaces, special characters)
original_columns = df.columns.tolist()
df.columns = [col.strip().replace(' ', '_').replace('(', '').replace(')', '').replace('%', 'Pct')
              for col in df.columns]

print(f"\n📝 COLUMN STANDARDIZATION")
print("=" * 50)
for orig, new in zip(original_columns, df.columns):
    if orig != new:
        print(f"'{orig}' → '{new}'")

# Display transformed dataset info
print(f"\n📊 TRANSFORMED DATASET")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Check data types
print(f"\n🔍 DATA TYPES")
print("=" * 50)
print(df.dtypes)

# Display sample of transformed data
print(f"\n👁️  TRANSFORMED DATA SAMPLE")
print("=" * 50)
print(df.head(10))

print(f"\n✅ Data transformation completed!")

🔄 DATA TRANSFORMATION
Original columns with potential time data:
Time candidates: ['Date']
✅ Created TimeIndex column: 1 to 165672 minutes
✅ Dropped original time columns: ['Date']

📝 COLUMN STANDARDIZATION

📊 TRANSFORMED DATASET
Shape: (165672, 8)
Columns: ['InletTemp', 'InletFlow', 'OutletTemp', 'AirFuelRatio', 'FuelGasFlow', 'ExcessO2', 'FuelGasMW', 'TimeIndex']

🔍 DATA TYPES
InletTemp       float64
InletFlow       float64
OutletTemp      float64
AirFuelRatio    float64
FuelGasFlow     float64
ExcessO2        float64
FuelGasMW       float64
TimeIndex         int64
dtype: object

👁️  TRANSFORMED DATA SAMPLE
   InletTemp  InletFlow  OutletTemp  AirFuelRatio  FuelGasFlow  ExcessO2  \
0 261.921478 165.248749  289.977783     11.498828   588.381287  2.299289   
1 261.911652 164.872009  290.056610     11.517564   586.792297  2.292834   
2 261.878510 164.925217  290.141907     11.536301   584.424438  2.336514   
3 261.934540 164.916031  290.285034     11.555037   582.956848  2.293091   
4 2

## 2. Physics-Informed Feature Engineering

Calculate advanced combustion parameters using the molecular weight data to create physics-informed features for better furnace modeling.

In [15]:
# Physics-Informed Feature Engineering with Molecular Weight
print("🔬 PHYSICS-INFORMED FEATURE ENGINEERING")
print("=" * 60)

# Constants for calculations
R_GAS = 8.314  # Universal gas constant (J/mol·K)
P_STANDARD = 101325  # Standard pressure (Pa)
T_STANDARD = 288.15  # Standard temperature (K, 15°C)
AIR_MW = 28.97  # Molecular weight of air (g/mol)

# Identify the molecular weight column
mw_column = None
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['mw', 'molecular', 'weight']):
        mw_column = col
        break

if mw_column:
    print(f"✅ Found molecular weight column: {mw_column}")
    MW_fuel = df[mw_column]
else:
    print("⚠️  Molecular weight column not found. Using typical natural gas MW = 16.5 g/mol")
    MW_fuel = 16.5  # Typical natural gas MW

# Identify other required columns (adapt based on actual column names)
fuel_flow_col = None
inlet_temp_col = None
afr_col = None
outlet_temp_col = None
excess_o2_col = None

# Search for column patterns
column_patterns = {
    'fuel_flow': ['fuel', 'flow', 'gas'],
    'inlet_temp': ['inlet', 'temp', 'temperature'],
    'afr': ['afr', 'air', 'fuel', 'ratio'],
    'outlet_temp': ['outlet', 'temp', 'temperature'],
    'excess_o2': ['excess', 'o2', 'oxygen']
}

print(f"\n🔍 IDENTIFYING PROCESS COLUMNS")
print("=" * 50)

identified_columns = {}
for pattern_name, keywords in column_patterns.items():
    for col in df.columns:
        if all(keyword.lower() in col.lower() for keyword in keywords):
            identified_columns[pattern_name] = col
            print(f"✅ {pattern_name}: {col}")
            break
        elif any(keyword.lower() in col.lower() for keyword in keywords):
            # Partial match - need manual verification
            identified_columns[pattern_name] = col
            print(f"⚠️  {pattern_name} (partial match): {col}")
            break

# If columns not found, show available columns for manual selection
if len(identified_columns) < 4:
    print(f"\n📋 AVAILABLE COLUMNS FOR MANUAL MAPPING:")
    for i, col in enumerate(df.columns, 1):
        print(f"{i:2d}. {col}")

# Calculate physics-informed features
print(f"\n⚡ CALCULATING PHYSICS-INFORMED FEATURES")
print("=" * 60)

# 1. Fuel Gas Density (at standard conditions initially)
# ρ = (P × MW) / (R × T)
df['FuelDensity_std'] = (P_STANDARD * MW_fuel) / (R_GAS * T_STANDARD)
print(f"✅ Calculated standard fuel density")

# 2. Specific Gravity (relative to air)
df['SpecificGravity'] = MW_fuel / AIR_MW
print(f"✅ Calculated specific gravity")

# 3. Approximate Wobbe Index (assuming typical heating value)
# For natural gas: HV ≈ 37-40 MJ/m³, we'll use 38 MJ/m³
heating_value_approx = 38.0  # MJ/m³
df['WobbeIndex_approx'] = heating_value_approx / np.sqrt(df['SpecificGravity'])
print(f"✅ Calculated approximate Wobbe Index")

# 4. If we have fuel flow, calculate mass flow rate
if 'fuel_flow' in identified_columns:
    fuel_flow_col = identified_columns['fuel_flow']
    df['FuelMassFlow'] = df[fuel_flow_col] * df['FuelDensity_std']
    df['ActualHeatInput'] = df['FuelMassFlow'] * heating_value_approx
    print(f"✅ Calculated fuel mass flow and heat input")

# 5. Stoichiometric calculations (for typical natural gas composition)
# Assuming CH₄ dominant: CH₄ + 2O₂ → CO₂ + 2H₂O
# Stoichiometric AFR for methane ≈ 17.2
stoich_afr_base = 17.2
df['StoichAFR'] = stoich_afr_base * (16.04 / MW_fuel)  # Adjust for actual MW
print(f"✅ Calculated stoichiometric AFR")

# 6. If we have AFR data, calculate excess air and efficiency metrics
if 'afr' in identified_columns:
    afr_col = identified_columns['afr']
    df['ExcessAirRatio'] = df[afr_col] / df['StoichAFR'] - 1
    df['CombustionEfficiency'] = 1 / (1 + 0.1 * df['ExcessAirRatio'])  # Simplified model
    print(f"✅ Calculated excess air ratio and combustion efficiency")

# 7. Thermal parameters
if 'inlet_temp' in identified_columns and 'outlet_temp' in identified_columns:
    inlet_temp_col = identified_columns['inlet_temp']
    outlet_temp_col = identified_columns['outlet_temp']
    df['TempRise'] = df[outlet_temp_col] - df[inlet_temp_col]
    if 'ActualHeatInput' in df.columns:
        df['ThermalEfficiency'] = df['TempRise'] / df['ActualHeatInput'] * 1000  # Simplified
    print(f"✅ Calculated thermal parameters")

# Display new features
new_features = [col for col in df.columns if col not in df_raw.columns and col != 'TimeIndex']
print(f"\n📊 NEW PHYSICS-INFORMED FEATURES")
print("=" * 60)
for i, feature in enumerate(new_features, 1):
    print(f"{i:2d}. {feature}")

# Statistical summary of new features
if new_features:
    print(f"\n📈 FEATURE STATISTICS")
    print("=" * 60)
    print(df[new_features].describe())

print(f"\n✅ Physics-informed feature engineering completed!")
print(f"Total features: {len(df.columns)} (added {len(new_features)} new features)")

🔬 PHYSICS-INFORMED FEATURE ENGINEERING
✅ Found molecular weight column: FuelGasMW

🔍 IDENTIFYING PROCESS COLUMNS
⚠️  fuel_flow (partial match): InletFlow
⚠️  inlet_temp (partial match): InletTemp
⚠️  afr (partial match): AirFuelRatio
⚠️  outlet_temp (partial match): InletTemp
⚠️  excess_o2 (partial match): ExcessO2

⚡ CALCULATING PHYSICS-INFORMED FEATURES
✅ Calculated standard fuel density
✅ Calculated specific gravity
✅ Calculated approximate Wobbe Index
✅ Calculated fuel mass flow and heat input
✅ Calculated stoichiometric AFR
✅ Calculated excess air ratio and combustion efficiency
✅ Calculated thermal parameters

📊 NEW PHYSICS-INFORMED FEATURES
 1. FuelDensity_std
 2. SpecificGravity
 3. WobbeIndex_approx
 4. FuelMassFlow
 5. ActualHeatInput
 6. StoichAFR
 7. ExcessAirRatio
 8. CombustionEfficiency
 9. TempRise
10. ThermalEfficiency

📈 FEATURE STATISTICS
       FuelDensity_std  SpecificGravity  WobbeIndex_approx  FuelMassFlow  \
count    165672.000000    165672.000000      165672.00

## 3. Final Data Preparation and Export

Prepare the enhanced dataset with physics-informed features for improved LNN model training and create train/validation/test splits.

In [16]:
# Final Data Preparation for Enhanced LNN Model
print("🎯 FINAL DATA PREPARATION FOR LNN TRAINING")
print("=" * 70)

# Define input and output features for LNN model
print(f"📋 FEATURE SELECTION")
print("=" * 50)

# Input features (enhanced with physics-informed features)
input_features = [
    # Original process variables
    'InletTemp', 'InletFlow', 'AirFuelRatio', 'FuelGasFlow',
    # Physics-informed features
    'FuelDensity_std', 'SpecificGravity', 'WobbeIndex_approx', 
    'StoichAFR', 'FuelGasMW'
]

# Add conditional features if they exist
if 'FuelMassFlow' in df.columns:
    input_features.append('FuelMassFlow')
if 'ActualHeatInput' in df.columns:
    input_features.append('ActualHeatInput')
if 'ExcessAirRatio' in df.columns:
    input_features.append('ExcessAirRatio')
if 'CombustionEfficiency' in df.columns:
    input_features.append('CombustionEfficiency')

# Output features (targets for prediction)
output_features = ['OutletTemp', 'ExcessO2']

print(f"Input features ({len(input_features)}):")
for i, feat in enumerate(input_features, 1):
    print(f"{i:2d}. {feat}")

print(f"\nOutput features ({len(output_features)}):")
for i, feat in enumerate(output_features, 1):
    print(f"{i:2d}. {feat}")

# Verify all features exist in dataframe
missing_features = [feat for feat in input_features + output_features if feat not in df.columns]
if missing_features:
    print(f"\n⚠️  Missing features: {missing_features}")
    # Remove missing features
    input_features = [feat for feat in input_features if feat in df.columns]
    output_features = [feat for feat in output_features if feat in df.columns]
    print(f"Adjusted input features: {input_features}")
    print(f"Adjusted output features: {output_features}")

# Create feature matrix
X_data = df[input_features].values
y_data = df[output_features].values

print(f"\n📊 DATA MATRIX SHAPES")
print("=" * 50)
print(f"Input matrix (X): {X_data.shape}")
print(f"Output matrix (y): {y_data.shape}")
print(f"Total samples: {len(X_data):,}")

# Create sequence data for LSTM (using sliding window)
def create_sequences(X, y, sequence_length=10):
    """Create sequences for LSTM training."""
    X_sequences = []
    y_sequences = []
    
    for i in range(sequence_length, len(X)):
        X_sequences.append(X[i-sequence_length:i])
        y_sequences.append(y[i])
    
    return np.array(X_sequences), np.array(y_sequences)

sequence_length = 10
print(f"\n🔄 CREATING SEQUENCES (sequence_length={sequence_length})")
print("=" * 60)

X_sequences, y_sequences = create_sequences(X_data, y_data, sequence_length)

print(f"Sequence shapes:")
print(f"X_sequences: {X_sequences.shape} (samples, timesteps, features)")
print(f"y_sequences: {y_sequences.shape} (samples, targets)")
print(f"Sequences created: {len(X_sequences):,}")

# Train/Validation/Test split
print(f"\n✂️  TRAIN/VALIDATION/TEST SPLIT")
print("=" * 50)

# 70% train, 15% validation, 15% test
train_size = 0.7
val_size = 0.15
test_size = 0.15

n_samples = len(X_sequences)
n_train = int(train_size * n_samples)
n_val = int(val_size * n_samples)
n_test = n_samples - n_train - n_val

# Split sequentially (preserving temporal order)
X_train = X_sequences[:n_train]
y_train = y_sequences[:n_train]

X_val = X_sequences[n_train:n_train+n_val]
y_val = y_sequences[n_train:n_train+n_val]

X_test = X_sequences[n_train+n_val:]
y_test = y_sequences[n_train+n_val:]

print(f"Training set: {X_train.shape[0]:,} samples ({train_size*100:.0f}%)")
print(f"Validation set: {X_val.shape[0]:,} samples ({val_size*100:.0f}%)")
print(f"Test set: {X_test.shape[0]:,} samples ({test_size*100:.0f}%)")

# Normalize the data
print(f"\n🔧 DATA NORMALIZATION")
print("=" * 50)

# Fit scalers on training data only
input_scaler = MinMaxScaler()
output_scaler = MinMaxScaler()

# Reshape for scaling
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
X_train_scaled_flat = input_scaler.fit_transform(X_train_flat)
X_train_scaled = X_train_scaled_flat.reshape(X_train.shape)

y_train_scaled = output_scaler.fit_transform(y_train)

# Apply scaling to validation and test sets
X_val_flat = X_val.reshape(-1, X_val.shape[-1])
X_val_scaled_flat = input_scaler.transform(X_val_flat)
X_val_scaled = X_val_scaled_flat.reshape(X_val.shape)

X_test_flat = X_test.reshape(-1, X_test.shape[-1])
X_test_scaled_flat = input_scaler.transform(X_test_flat)
X_test_scaled = X_test_scaled_flat.reshape(X_test.shape)

y_val_scaled = output_scaler.transform(y_val)
y_test_scaled = output_scaler.transform(y_test)

print(f"✅ Data normalization completed")
print(f"Input scaling range: [0, 1]")
print(f"Output scaling range: [0, 1]")

# Data summary
print(f"\n📈 ENHANCED DATASET SUMMARY")
print("=" * 60)
print(f"Dataset: BenchmarkData2 with physics-informed features")
print(f"Total original rows: {len(df_raw):,}")
print(f"Total sequences: {len(X_sequences):,}")
print(f"Sequence length: {sequence_length} timesteps")
print(f"Input features: {len(input_features)} (including MW-derived features)")
print(f"Output targets: {len(output_features)}")
print(f"Training samples: {len(X_train_scaled):,}")
print(f"Validation samples: {len(X_val_scaled):,}")
print(f"Test samples: {len(X_test_scaled):,}")

# Save prepared data
import pickle

enhanced_training_data = {
    'X_train': X_train_scaled,
    'y_train': y_train_scaled,
    'X_val': X_val_scaled,
    'y_val': y_val_scaled,
    'X_test': X_test_scaled,
    'y_test': y_test_scaled,
    'input_features': input_features,
    'output_features': output_features,
    'normalization_params': {
        'input_scaler': input_scaler,
        'output_scaler': output_scaler
    },
    'data_summary': {
        'total_samples': len(X_sequences),
        'sequence_length': sequence_length,
        'n_input_features': len(input_features),
        'n_output_features': len(output_features),
        'train_samples': len(X_train_scaled),
        'val_samples': len(X_val_scaled),
        'test_samples': len(X_test_scaled)
    }
}

# Save to file
output_file = '/Users/abuhuzaifahbidin/Documents/GitHub/furnace-commander/backend/data/enhanced_training_data.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(enhanced_training_data, f)

print(f"\n💾 ENHANCED DATASET SAVED")
print("=" * 50)
print(f"File: enhanced_training_data.pkl")
print(f"Location: {output_file}")
print(f"Ready for improved LNN model training!")

print(f"\n🎉 DATA PREPARATION COMPLETED!")
print("=" * 60)
print("✅ Physics-informed features created using molecular weight")
print("✅ Data cleaned and validated") 
print("✅ Sequences created for LSTM training")
print("✅ Train/validation/test splits prepared")
print("✅ Data normalized and saved")
print("🚀 Ready for enhanced LNN model with significantly improved features!")

🎯 FINAL DATA PREPARATION FOR LNN TRAINING
📋 FEATURE SELECTION
Input features (13):
 1. InletTemp
 2. InletFlow
 3. AirFuelRatio
 4. FuelGasFlow
 5. FuelDensity_std
 6. SpecificGravity
 7. WobbeIndex_approx
 8. StoichAFR
 9. FuelGasMW
10. FuelMassFlow
11. ActualHeatInput
12. ExcessAirRatio
13. CombustionEfficiency

Output features (2):
 1. OutletTemp
 2. ExcessO2

📊 DATA MATRIX SHAPES
Input matrix (X): (165672, 13)
Output matrix (y): (165672, 2)
Total samples: 165,672

🔄 CREATING SEQUENCES (sequence_length=10)
Sequence shapes:
X_sequences: (165662, 10, 13) (samples, timesteps, features)
y_sequences: (165662, 2) (samples, targets)
Sequences created: 165,662

✂️  TRAIN/VALIDATION/TEST SPLIT
Training set: 115,963 samples (70%)
Validation set: 24,849 samples (15%)
Test set: 24,850 samples (15%)

🔧 DATA NORMALIZATION
✅ Data normalization completed
Input scaling range: [0, 1]
Output scaling range: [0, 1]

📈 ENHANCED DATASET SUMMARY
Dataset: BenchmarkData2 with physics-informed features
Total 

In [19]:
# Domain-Driven Outlier Detection - Temporal Consistency Analysis
print("🔍 DOMAIN-DRIVEN OUTLIER DETECTION")
print("=" * 70)
print("Using temporal consistency to identify sudden, non-physical deviations...")

# Create a copy for outlier detection
df_temporal = df.copy()

# Define process variables for temporal analysis
process_variables = [
    'InletTemp', 'InletFlow', 'OutletTemp', 'AirFuelRatio', 
    'FuelGasFlow', 'ExcessO2', 'FuelGasMW'
]

# Remove variables that don't exist in our dataset
process_variables = [var for var in process_variables if var in df_temporal.columns]
print(f"Analyzing {len(process_variables)} process variables for temporal consistency:")
for var in process_variables:
    print(f"  - {var}")

# Method 1: Absolute Difference Threshold
print(f"\n📊 METHOD 1: ABSOLUTE DIFFERENCE ANALYSIS")
print("=" * 60)

outlier_indices_abs = set()

# Calculate absolute differences and detect outliers
for var in process_variables:
    # Calculate absolute difference between consecutive points
    abs_diff = np.abs(df_temporal[var].diff())
    
    # Calculate statistics for this variable
    median_diff = abs_diff.median()
    q75_diff = abs_diff.quantile(0.75)
    q95_diff = abs_diff.quantile(0.95)
    
    # Define threshold as multiple of 75th percentile (more robust than mean)
    # This captures sudden jumps that are much larger than typical variations
    threshold_conservative = q75_diff * 3  # Conservative: 3x 75th percentile
    threshold_aggressive = q75_diff * 2    # Aggressive: 2x 75th percentile
    
    # Use conservative threshold by default
    threshold = threshold_conservative
    
    # Find outliers
    outliers = abs_diff > threshold
    outlier_count = outliers.sum()
    
    print(f"\n{var}:")
    print(f"  Median diff: {median_diff:.4f}")
    print(f"  75th percentile: {q75_diff:.4f}")
    print(f"  95th percentile: {q95_diff:.4f}")
    print(f"  Threshold (3x Q75): {threshold:.4f}")
    print(f"  Outliers detected: {outlier_count}")
    
    if outlier_count > 0:
        # Get indices of outliers (excluding first row since diff[0] is NaN)
        var_outlier_indices = df_temporal.index[outliers & ~abs_diff.isna()]
        outlier_indices_abs.update(var_outlier_indices)
        
        # Show some examples
        if len(var_outlier_indices) > 0:
            sample_outliers = var_outlier_indices[:3]  # First 3 outliers
            print(f"  Sample outlier indices: {list(sample_outliers)}")
            for idx in sample_outliers:
                if idx > 0:  # Ensure we can show previous value
                    prev_val = df_temporal.loc[idx-1, var]
                    curr_val = df_temporal.loc[idx, var]
                    diff_val = abs(curr_val - prev_val)
                    print(f"    Row {idx}: {prev_val:.2f} → {curr_val:.2f} (Δ={diff_val:.2f})")

print(f"\nTotal unique outlier indices (Method 1): {len(outlier_indices_abs)}")

# Method 2: Statistical Standard Deviation Analysis
print(f"\n📈 METHOD 2: STATISTICAL DEVIATION ANALYSIS")
print("=" * 60)

outlier_indices_std = set()

for var in process_variables:
    # Calculate rolling statistics for more robust outlier detection
    window_size = min(50, len(df_temporal) // 10)  # Adaptive window size
    
    # Calculate differences
    diff_series = df_temporal[var].diff()
    
    # Calculate rolling mean and std of differences
    rolling_mean = diff_series.rolling(window=window_size, center=True).mean()
    rolling_std = diff_series.rolling(window=window_size, center=True).std()
    
    # Z-score for differences (how many standard deviations from rolling mean)
    z_scores = np.abs((diff_series - rolling_mean) / rolling_std)
    
    # Threshold: points more than 3 standard deviations from rolling mean
    z_threshold = 3.0
    outliers_z = z_scores > z_threshold
    outlier_count_z = outliers_z.sum()
    
    print(f"\n{var}:")
    print(f"  Rolling window size: {window_size}")
    print(f"  Mean absolute diff: {np.abs(diff_series).mean():.4f}")
    print(f"  Std of diffs: {diff_series.std():.4f}")
    print(f"  Z-score threshold: {z_threshold}")
    print(f"  Outliers detected: {outlier_count_z}")
    
    if outlier_count_z > 0:
        var_outlier_indices_z = df_temporal.index[outliers_z & ~z_scores.isna()]
        outlier_indices_std.update(var_outlier_indices_z)

print(f"\nTotal unique outlier indices (Method 2): {len(outlier_indices_std)}")

# Method 3: Combined Approach
print(f"\n🎯 METHOD 3: COMBINED ROBUST APPROACH")
print("=" * 60)

# Combine both methods - flag as outlier if detected by either method
combined_outliers = outlier_indices_abs.union(outlier_indices_std)

print(f"Method 1 (Absolute): {len(outlier_indices_abs)} outliers")
print(f"Method 2 (Statistical): {len(outlier_indices_std)} outliers")
print(f"Combined (Union): {len(combined_outliers)} outliers")
print(f"Overlap: {len(outlier_indices_abs.intersection(outlier_indices_std))} outliers")

# Analyze outlier distribution
if len(combined_outliers) > 0:
    outlier_list = sorted(list(combined_outliers))
    print(f"\nOutlier distribution:")
    print(f"  First outlier at index: {outlier_list[0]}")
    print(f"  Last outlier at index: {outlier_list[-1]}")
    print(f"  Percentage of data: {len(combined_outliers)/len(df_temporal)*100:.2f}%")

# Choose the method to use (let's use Method 1 - absolute difference as it's more conservative)
chosen_outliers = outlier_indices_abs
print(f"\n✅ CHOSEN METHOD: Absolute Difference (Method 1)")
print(f"Selected outliers: {len(chosen_outliers)}")

# Create cleaned dataset by removing outliers
print(f"\n🧹 CREATING CLEANED DATASET")
print("=" * 60)

df_cleaned_temporal = df_temporal.drop(index=chosen_outliers).reset_index(drop=True)

print(f"Original dataset: {len(df_temporal):,} rows")
print(f"Outliers removed: {len(chosen_outliers):,} rows")
print(f"Cleaned dataset: {len(df_cleaned_temporal):,} rows")
print(f"Data retention: {len(df_cleaned_temporal)/len(df_temporal)*100:.1f}%")

# Update the main dataframe for next steps
df = df_cleaned_temporal.copy()

# Verify data quality after cleaning
print(f"\n📊 DATA QUALITY AFTER TEMPORAL CLEANING")
print("=" * 60)

for var in process_variables:
    if var in df.columns:
        # Recalculate difference statistics
        new_diff = np.abs(df[var].diff())
        median_new = new_diff.median()
        q95_new = new_diff.quantile(0.95)
        max_new = new_diff.max()
        
        print(f"{var}:")
        print(f"  Median abs diff: {median_new:.4f}")
        print(f"  95th percentile: {q95_new:.4f}")
        print(f"  Maximum diff: {max_new:.4f}")

print(f"\n✅ Domain-driven outlier detection completed!")
print(f"🚀 Dataset is now cleaned of sudden, non-physical deviations!")
print(f"💡 This should significantly reduce noise-driven overfitting in the LNN model!")

🔍 DOMAIN-DRIVEN OUTLIER DETECTION
Using temporal consistency to identify sudden, non-physical deviations...
Analyzing 7 process variables for temporal consistency:
  - InletTemp
  - InletFlow
  - OutletTemp
  - AirFuelRatio
  - FuelGasFlow
  - ExcessO2
  - FuelGasMW

📊 METHOD 1: ABSOLUTE DIFFERENCE ANALYSIS

InletTemp:
  Median diff: 0.0313
  75th percentile: 0.0573
  95th percentile: 0.1157
  Threshold (3x Q75): 0.1719
  Outliers detected: 1753
  Sample outlier indices: [23, 118, 198]
    Row 23: 262.10 → 261.93 (Δ=0.18)
    Row 118: 261.51 → 261.71 (Δ=0.20)
    Row 198: 261.38 → 261.58 (Δ=0.20)

InletFlow:
  Median diff: 0.1364
  75th percentile: 0.2464
  95th percentile: 0.4623
  Threshold (3x Q75): 0.7391
  Outliers detected: 938
  Sample outlier indices: [396, 1079, 1730]
    Row 396: 165.59 → 164.70 (Δ=0.89)
    Row 1079: 172.43 → 171.63 (Δ=0.79)
    Row 1730: 171.90 → 172.67 (Δ=0.77)

OutletTemp:
  Median diff: 0.0566
  75th percentile: 0.1014
  95th percentile: 0.1862
  Thresho

In [20]:
# Summary of Temporal Cleaning Results and Feature Engineering
print("📋 CLEANING SUMMARY")
print("=" * 50)
print(f"Final dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nData types:")
print(df.dtypes)

print(f"\n📊 BASIC STATISTICS AFTER CLEANING")
print("=" * 50)
print(df.describe())

# Continue with the feature engineering using the cleaned dataset
print(f"\n🔧 FEATURE ENGINEERING ON CLEANED DATA")
print("=" * 50)

# Re-engineer features on the cleaned dataset
# Calculate fuel gas density (kg/m³) using ideal gas law
# ρ = PM/RT where P=pressure, M=molecular weight, R=gas constant, T=temperature
R_gas = 8.314  # Universal gas constant (J/mol·K)
P_atm = 101325  # Atmospheric pressure (Pa)
T_std = 288.15  # Standard temperature (K, 15°C)

df['FuelGasDensity'] = (P_atm * df['FuelGasMW']) / (R_gas * T_std)
print(f"✓ Calculated fuel gas density (range: {df['FuelGasDensity'].min():.3f} - {df['FuelGasDensity'].max():.3f} kg/m³)")

# Specific gravity (relative to air, MW_air ≈ 28.97)
MW_air = 28.97
df['SpecificGravity'] = df['FuelGasMW'] / MW_air
print(f"✓ Calculated specific gravity (range: {df['SpecificGravity'].min():.3f} - {df['SpecificGravity'].max():.3f})")

# Wobbe Index (measure of fuel interchangeability)
# Higher heating value estimation based on molecular weight
# Simple correlation: HHV ≈ 50 MJ/kg for typical fuel gases
HHV_est = 50e6  # J/kg (rough estimate)
df['WobbeIndex'] = HHV_est / np.sqrt(df['SpecificGravity'])
print(f"✓ Calculated Wobbe Index (range: {df['WobbeIndex'].min():.0f} - {df['WobbeIndex'].max():.0f} J/m³)")

# Stoichiometric air-fuel ratio calculation
# Rough estimate: AFR_stoich ≈ 17.2 for natural gas (adjust based on MW)
AFR_stoich_base = 17.2
df['AFR_Stoichiometric'] = AFR_stoich_base * (df['FuelGasMW'] / 16.04)  # 16.04 is MW of methane
print(f"✓ Calculated stoichiometric AFR (range: {df['AFR_Stoichiometric'].min():.2f} - {df['AFR_Stoichiometric'].max():.2f})")

# Fuel mass flow rate (kg/s) 
# Mass flow = Volume flow × Density
df['FuelMassFlow'] = df['FuelGasFlow'] * df['FuelGasDensity'] / 3600  # Convert from m³/h to kg/s
print(f"✓ Calculated fuel mass flow (range: {df['FuelMassFlow'].min():.4f} - {df['FuelMassFlow'].max():.4f} kg/s)")

# Actual heat input (MW)
df['ActualHeatInput'] = df['FuelMassFlow'] * HHV_est / 1e6  # Convert to MW
print(f"✓ Calculated actual heat input (range: {df['ActualHeatInput'].min():.2f} - {df['ActualHeatInput'].max():.2f} MW)")

# Excess air ratio
df['ExcessAirRatio'] = df['AirFuelRatio'] / df['AFR_Stoichiometric']
print(f"✓ Calculated excess air ratio (range: {df['ExcessAirRatio'].min():.3f} - {df['ExcessAirRatio'].max():.3f})")

# Combustion efficiency estimation (based on excess O2)
# Simplified correlation: lower excess O2 generally means better efficiency (up to a point)
df['CombustionEfficiency'] = 100 - (df['ExcessO2'] * 2)  # Simplified model
df['CombustionEfficiency'] = np.clip(df['CombustionEfficiency'], 70, 99)  # Reasonable bounds
print(f"✓ Calculated combustion efficiency (range: {df['CombustionEfficiency'].min():.1f}% - {df['CombustionEfficiency'].max():.1f}%)")

# Temperature difference (process indicator)
df['TempRise'] = df['OutletTemp'] - df['InletTemp']
print(f"✓ Calculated temperature rise (range: {df['TempRise'].min():.1f}°C - {df['TempRise'].max():.1f}°C)")

# Thermal efficiency (simplified)
# Assuming constant specific heat and using temperature rise
specific_heat_air = 1005  # J/kg·K for air
air_density = 1.225  # kg/m³ at standard conditions
df['ThermalEfficiency'] = (df['InletFlow'] * air_density * specific_heat_air * df['TempRise']) / (df['ActualHeatInput'] * 1e6) * 100
df['ThermalEfficiency'] = np.clip(df['ThermalEfficiency'], 0, 100)  # Bound between 0-100%
print(f"✓ Calculated thermal efficiency (range: {df['ThermalEfficiency'].min():.1f}% - {df['ThermalEfficiency'].max():.1f}%)")

print(f"\n📈 FINAL FEATURE SET")
print("=" * 50)
print(f"Total features: {len(df.columns)}")
print("Features:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n✅ Feature engineering completed on cleaned dataset!")
print(f"📊 Dataset shape: {df.shape}")
print(f"🚀 Ready for advanced LNN model training!")

📋 CLEANING SUMMARY
Final dataset shape: (154022, 18)
Columns: ['InletTemp', 'InletFlow', 'OutletTemp', 'AirFuelRatio', 'FuelGasFlow', 'ExcessO2', 'FuelGasMW', 'TimeIndex', 'FuelDensity_std', 'SpecificGravity', 'WobbeIndex_approx', 'FuelMassFlow', 'ActualHeatInput', 'StoichAFR', 'ExcessAirRatio', 'CombustionEfficiency', 'TempRise', 'ThermalEfficiency']

Data types:
InletTemp               float64
InletFlow               float64
OutletTemp              float64
AirFuelRatio            float64
FuelGasFlow             float64
ExcessO2                float64
FuelGasMW               float64
TimeIndex                 int64
FuelDensity_std         float64
SpecificGravity         float64
WobbeIndex_approx       float64
FuelMassFlow            float64
ActualHeatInput         float64
StoichAFR               float64
ExcessAirRatio          float64
CombustionEfficiency    float64
TempRise                float64
ThermalEfficiency       float64
dtype: object

📊 BASIC STATISTICS AFTER CLEANING
        

In [21]:
# Final Dataset Preparation - Sequences and Normalization
print("🎯 FINAL DATASET PREPARATION")
print("=" * 60)

# Define feature categories for organized processing
input_features = [
    # Original process variables
    'InletTemp', 'InletFlow', 'AirFuelRatio', 'FuelGasFlow', 'FuelGasMW',
    # Physics-informed features
    'FuelGasDensity', 'SpecificGravity', 'WobbeIndex', 'AFR_Stoichiometric',
    'FuelMassFlow', 'ActualHeatInput', 'ExcessAirRatio', 'CombustionEfficiency', 'ThermalEfficiency'
]

target_features = ['OutletTemp', 'ExcessO2', 'TempRise']

# Verify all features exist
input_features = [f for f in input_features if f in df.columns]
target_features = [f for f in target_features if f in df.columns]

print(f"Input features ({len(input_features)}):")
for i, feat in enumerate(input_features, 1):
    print(f"  {i:2d}. {feat}")

print(f"\nTarget features ({len(target_features)}):")
for i, feat in enumerate(target_features, 1):
    print(f"  {i:2d}. {feat}")

# Prepare data for sequence modeling
all_features = input_features + target_features
df_modeling = df[all_features].copy()

# Handle any remaining NaN values
print(f"\n🔍 HANDLING MISSING VALUES")
print("=" * 40)
nan_counts = df_modeling.isnull().sum()
total_nans = nan_counts.sum()

if total_nans > 0:
    print(f"Total NaN values found: {total_nans}")
    for col, nan_count in nan_counts[nan_counts > 0].items():
        print(f"  {col}: {nan_count} NaN values")
    
    # Forward fill then backward fill to handle NaNs
    df_modeling = df_modeling.fillna(method='ffill').fillna(method='bfill')
    print("✓ NaN values filled using forward/backward fill")
else:
    print("✓ No NaN values found")

# Normalize the data
print(f"\n📏 DATA NORMALIZATION")
print("=" * 40)

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Use StandardScaler for better handling of outliers
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# Fit scalers on the data
X_data = df_modeling[input_features].values
y_data = df_modeling[target_features].values

X_scaled = scaler_X.fit_transform(X_data)
y_scaled = scaler_y.fit_transform(y_data)

print(f"✓ Input features normalized: {X_scaled.shape}")
print(f"✓ Target features normalized: {y_scaled.shape}")

# Create sequences for LSTM/LNN
print(f"\n🔄 SEQUENCE CREATION")
print("=" * 40)

sequence_length = 10  # Look at 10 previous time steps

def create_sequences(X, y, seq_length):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:(i + seq_length)])
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

X_sequences, y_sequences = create_sequences(X_scaled, y_scaled, sequence_length)

print(f"✓ Sequences created:")
print(f"  Input sequences: {X_sequences.shape} (samples, timesteps, features)")
print(f"  Target sequences: {y_sequences.shape} (samples, targets)")
print(f"  Sequence length: {sequence_length} timesteps")

# Train/validation split (temporal split - use later data for validation)
split_ratio = 0.8
split_idx = int(len(X_sequences) * split_ratio)

X_train = X_sequences[:split_idx]
X_val = X_sequences[split_idx:]
y_train = y_sequences[:split_idx]
y_val = y_sequences[split_idx:]

print(f"\n📊 TRAIN/VALIDATION SPLIT")
print("=" * 40)
print(f"Training set: {X_train.shape[0]:,} sequences ({split_ratio*100:.0f}%)")
print(f"Validation set: {X_val.shape[0]:,} sequences ({(1-split_ratio)*100:.0f}%)")

# Save the processed data
print(f"\n💾 SAVING PROCESSED DATA")
print("=" * 40)

# Prepare the data package
processed_data = {
    'X_train': X_train,
    'X_val': X_val,
    'y_train': y_train,
    'y_val': y_val,
    'scaler_X': scaler_X,
    'scaler_y': scaler_y,
    'input_features': input_features,
    'target_features': target_features,
    'sequence_length': sequence_length,
    'original_data_shape': df_modeling.shape,
    'cleaning_info': {
        'outliers_removed': 'domain_driven_temporal_analysis',
        'cleaning_method': 'absolute_difference_threshold',
        'final_data_retention': f"{len(df_modeling)/len(df)*100:.1f}%"
    }
}

# Save to pickle
import pickle
save_path = 'data/enhanced_training_data_cleaned.pkl'

with open(save_path, 'wb') as f:
    pickle.dump(processed_data, f)

print(f"✅ Data saved to: {save_path}")
print(f"📦 Package contents:")
print(f"  - Training sequences: {X_train.shape}")
print(f"  - Validation sequences: {X_val.shape}")
print(f"  - Input features: {len(input_features)}")
print(f"  - Target features: {len(target_features)}")
print(f"  - Scalers and metadata included")

# Data quality summary
print(f"\n📋 FINAL DATA QUALITY SUMMARY")
print("=" * 50)
print(f"✓ Domain-driven outlier detection completed")
print(f"✓ Temporal consistency validated")
print(f"✓ Physics-informed features engineered")
print(f"✓ Data normalized and sequenced")
print(f"✓ Train/validation split applied")
print(f"✓ Ready for enhanced LNN training!")

# Show some statistics
print(f"\nInput feature statistics (normalized):")
for i, feat in enumerate(input_features):
    feat_data = X_scaled[:, i]
    print(f"  {feat}: mean={feat_data.mean():.3f}, std={feat_data.std():.3f}")

print(f"\nTarget feature statistics (normalized):")
for i, feat in enumerate(target_features):
    feat_data = y_scaled[:, i]
    print(f"  {feat}: mean={feat_data.mean():.3f}, std={feat_data.std():.3f}")

print(f"\n🚀 Dataset is now optimally prepared for LNN training!")
print(f"🎯 Reduced overfitting risk through domain-driven cleaning!")
print(f"💡 Physics-informed features should improve model understanding!")

🎯 FINAL DATASET PREPARATION
Input features (14):
   1. InletTemp
   2. InletFlow
   3. AirFuelRatio
   4. FuelGasFlow
   5. FuelGasMW
   6. FuelGasDensity
   7. SpecificGravity
   8. WobbeIndex
   9. AFR_Stoichiometric
  10. FuelMassFlow
  11. ActualHeatInput
  12. ExcessAirRatio
  13. CombustionEfficiency
  14. ThermalEfficiency

Target features (3):
   1. OutletTemp
   2. ExcessO2
   3. TempRise

🔍 HANDLING MISSING VALUES
✓ No NaN values found

📏 DATA NORMALIZATION
✓ Input features normalized: (154022, 14)
✓ Target features normalized: (154022, 3)

🔄 SEQUENCE CREATION
✓ Sequences created:
  Input sequences: (154012, 10, 14) (samples, timesteps, features)
  Target sequences: (154012, 3) (samples, targets)
  Sequence length: 10 timesteps

📊 TRAIN/VALIDATION SPLIT
Training set: 123,209 sequences (80%)
Validation set: 30,803 sequences (20%)

💾 SAVING PROCESSED DATA
✅ Data saved to: data/enhanced_training_data_cleaned.pkl
📦 Package contents:
  - Training sequences: (123209, 10, 14)
  - Va